# Baseline Model (tf-idf with logistic regression)

A simple but strong baseline for the Smart MCQ Solver challenge.

**Pipeline:** raw text → TF-IDF vectorization → Logistic Regression → MAP@3 evaluation

I try three text-building strategies and pick the best by validation MAP@3:
1. **v1_simple** — prompt + all 5 options concatenated (naive baseline)
2. **v2_repeated** — prompt repeated before EACH option (gives TF-IDF more Q-A signal)
3. **v3_labeled** — explicit `option a: ...` labels + trigrams (helps TF-IDF differentiate positions)

All experiments are logged to **Weights & Biases** for reproducibility.

## 1. Imports & Paths

In [1]:
import pandas as pd
import numpy as np
import pickle
import wandb
from pathlib import Path

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [4]:
OUTPUT_DIR = Path('../outputs')
DATA_DIR   = Path('../data')

## 2. MAP@3 metric + helper functions

**MAP@3** (Mean Average Precision @ 3) is the competition metric:

| Position of correct answer | Score |
|----------------------------|-------|
| 1st                        | 1.00  |
| 2nd                        | 0.50  |
| 3rd                        | 0.33  |
| Not in top-3               | 0.00  |

We also define `make_submission()` and `evaluate()` here so they can be
reused across every experiment in the notebook.

In [5]:
ANSWER_MAP  = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
REVERSE_MAP = {v: k for k, v in ANSWER_MAP.items()}

In [6]:
"""
MAP@3: pos 1→1.0, 
       pos 2→0.5, 
       pos 3→0.33,
       miss→0.0
"""
def map_at_3(y_true, y_proba):
    scores = []
    for i, true in enumerate(y_true):
        # Top-3 predicted class indices (highest probability first)
        top3 = np.argsort(y_proba[i])[-3:][::-1]
        # Score = 1 / rank if correct answer is in top-3 else 0
        score = (1.0 / (np.where(top3 == true)[0][0] + 1) if true in top3 else 0.0) 
        scores.append(score)
    return float(np.mean(scores))

In [7]:
# Generate a Kaggle-format submission: 'ID,Prediction' with top-3 letters
def make_submission(model, vectorizer, test_df, test_ids):
    X = vectorizer.transform(test_df['combined_text'].values)
    proba = model.predict_proba(X)

    # Pick top-3 classes per row, then map indices back to A/B/C/D/E
    preds = []
    for i in range(len(test_df)):
        top3 = np.argsort(proba[i])[-3:][::-1]
        preds.append(' '.join([REVERSE_MAP[j] for j in top3]))
        
    return pd.DataFrame({'ID': test_ids, 'Prediction': preds})

In [8]:
# Compute accuracy, weighted-F1 and MAP@3 on a given split and print them.
def evaluate(model, vectorizer, df, split_name='val'):
    X     = vectorizer.transform(df['combined_text'].values)
    y     = df['answer'].map(ANSWER_MAP).values
    pred  = model.predict(X)
    proba = model.predict_proba(X)
    
    acc   = accuracy_score(y, pred)
    f1    = f1_score(y, pred, average='weighted')
    m3    = map_at_3(y, proba)

    print(f'  [{split_name}] ACC={acc:.4f}  F1={f1:.4f}  MAP@3={m3:.4f}')
    return {'accuracy': acc, 'f1': f1, 'map_at_3': m3}

print(" Helpers ready")

 Helpers ready


## 3. Load raw data

We read the competition CSVs and lowercase/strip every text column so that
TF-IDF doesn't treat `Photosynthesis` and `photosynthesis` as two tokens.

In [9]:
raw_train = pd.read_csv(DATA_DIR / 'train.csv')
raw_test  = pd.read_csv(DATA_DIR / 'test.csv')

In [10]:
# Lowercase + strip whitespace for every text column
text_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']
for col in text_cols:
    raw_train[col] = raw_train[col].str.lower().str.strip()
    raw_test[col]  = raw_test[col].str.lower().str.strip()

print(f'Train: {raw_train.shape} | Test: {raw_test.shape}')

Train: (2000, 8) | Test: (500, 7)


In [11]:
raw_train.head(1)

,id,prompt,A,B,C,D,E,answer
0,1,pick the best possible answer: what is martin ...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...,B


## 4. Stratified 80/20 train/val split

We shuffle each answer class separately and take 80% for training, 20% for
validation. This keeps the per-class distribution identical in both splits,
which matters because MAP@3 is sensitive to per-class behaviour.

In [12]:
# Stratified split before building the combined_text
np.random.seed(42)
train_idx, val_idx = [], []
for ans in 'ABCDE':
    idx = raw_train[raw_train['answer'] == ans].index.tolist()
    np.random.shuffle(idx)
    cut = int(len(idx) * 0.8)
    train_idx += idx[:cut]
    val_idx   += idx[cut:]

train_df = raw_train.loc[train_idx].reset_index(drop=True)
val_df   = raw_train.loc[val_idx].reset_index(drop=True)
test_df  = raw_test.copy()

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print("train/val split done")

Train: 1599 | Val: 401 | Test: 500
train/val split done


## 5. Text Combination Strategies

Three different ways to merge the prompt + five options into a single string per row. We'll experiment with each to see which gives the best signal to the classifier.

In [13]:
def build_v1_simple(df):
    """Strategy 1: prompt + all options concatenated (baseline)"""
    return (df['prompt'] + ' ' +
            df['A'] + ' ' + df['B'] + ' ' +
            df['C'] + ' ' + df['D'] + ' ' + df['E']).values

In [14]:
def build_v2_repeated(df):
    """
    Strategy 2: prompt repeated before EACH option
    Gives model more context per option.
    Format: 'Q optA Q optB Q optC Q optD Q optE'
    """
    q = df['prompt']
    return (q+' '+df['A']+' '+q+' '+df['B']+' '+
            q+' '+df['C']+' '+q+' '+df['D']+' '+q+' '+df['E']).values

In [15]:
def build_v3_labeled(df):
    """
    Strategy 3: label each option explicitly
    Format: 'Q option a: optA option b: optB ...'
    Helps TF-IDF differentiate option positions.
    """
    return (df['prompt'] +
            ' option a: ' + df['A'] +
            ' option b: ' + df['B'] +
            ' option c: ' + df['C'] +
            ' option d: ' + df['D'] +
            ' option e: ' + df['E']).values

In [16]:
# Apply v1 by default; the experiment runner will overwrite per-config.
for df in [train_df, val_df, test_df]:
    df['combined_text'] = build_v1_simple(df)

print('Text builder functions ready!')

Text builder functions ready!


In [17]:
print(f'\nExample (v1): {train_df["combined_text"].iloc[0][:100]}...')


Example (v1): which of the following is correct? which of the following is an accurate definition of dynamic scali...


## 6. Experiment runner

A single function that runs the full pipeline for a given config:

1. Build text using the chosen strategy
2. Fit TF-IDF on train, transform val + test
3. Train Logistic Regression
4. Evaluate on train + val (ACC / F1 / MAP@3)
5. Log metrics to WandB
6. Build a Kaggle submission file

Returns the trained `(model, tfidf, val_map3, submission)` tuple.

In [18]:
#  Full experiment: build text → vectorize → train → eval → log

def run_experiment(run_name, config, train_df, val_df, test_df):

    print(f'EXPERIMENT: {run_name}')

    # Build text based on strategy
    strategy = config.get('text_strategy', 'v1_simple')

    builder  = {'v1_simple'  : build_v1_simple,
                'v2_repeated': build_v2_repeated,
                'v3_labeled' : build_v3_labeled}[strategy]

    train_df = train_df.copy()
    val_df   = val_df.copy()
    test_df  = test_df.copy()
    train_df['combined_text'] = builder(train_df)
    val_df['combined_text']   = builder(val_df)
    test_df['combined_text']  = builder(test_df)
    print(f'  Text strategy : {strategy}')

    # TF-IDF
    tfidf = TfidfVectorizer(
        max_features = config['max_features'],
        min_df       = config['min_df'],
        max_df       = config['max_df'],
        ngram_range  = tuple(config['ngram_range']),
        sublinear_tf = True,
        strip_accents= 'unicode',
        analyzer     = 'word',
        token_pattern= r'\w{1,}',
        stop_words   = 'english'
    )

    X_train = tfidf.fit_transform(train_df['combined_text'].values)
    X_val   = tfidf.transform(val_df['combined_text'].values)
    X_test  = tfidf.transform(test_df['combined_text'].values)
    print(f'  TF-IDF features: {X_train.shape[1]}')

    # Labels
    y_train = train_df['answer'].map(ANSWER_MAP).values
    y_val   = val_df['answer'].map(ANSWER_MAP).values

    # Model
    model = LogisticRegression(
        C            = config['lr_C'],
        max_iter     = 1000,
        solver       = 'lbfgs',
        random_state = 42,
        n_jobs       = -1
    )
    model.fit(X_train, y_train)

    # Evaluate
    train_metrics = evaluate(model, tfidf, train_df, 'train')
    val_metrics   = evaluate(model, tfidf, val_df,   'val')

    # WandB
    wandb.login()
    run = wandb.init(
        project = '23f2003236-t22026',
        name    = run_name,
        config  = config,
        tags    = ['baseline'],
        reinit  = 'finish_previous'
    )
    wandb.log({
        'train/accuracy' : train_metrics['accuracy'],
        'train/f1'       : train_metrics['f1'],
        'train/map_at_3' : train_metrics['map_at_3'],
        'val/accuracy'   : val_metrics['accuracy'],
        'val/f1'         : val_metrics['f1'],
        'val/map_at_3'   : val_metrics['map_at_3'],
        'tfidf/features' : X_train.shape[1],
    })
    wandb.finish()
    print(f'  WandB run: {run_name}')

    # Submission
    sub = make_submission(model, tfidf, test_df, test_df['id'].values)

    return model, tfidf, val_metrics['map_at_3'], sub

print('Experiment runner ready')

Experiment runner ready


## 7. Experiment 1 — v1 baseline

The simplest possible setup: concatenate prompt + all options, 5K TF-IDF
features, unigrams + bigrams, `C=1.0`.

This is the floor — anything below this means something is broken haha.

In [19]:
config_v1 = {
    'text_strategy': 'v1_simple',
    'max_features' : 5000,
    'min_df'       : 5,
    'max_df'       : 0.8,
    'ngram_range'  : [1, 2],
    'lr_C'         : 1.0,
    'description'  : 'Baseline'
}

model_v1, tfidf_v1, val_map3_v1, sub_v1 = run_experiment(
    run_name = 'baseline_v1',
    config   = config_v1,
    train_df = train_df,
    val_df   = val_df,
    test_df  = test_df
)

sub_v1.to_csv(OUTPUT_DIR / 'predictions' / 'v1_submission.csv', index=False)
print(f'\nExperiment 1 done! Val MAP@3 = {val_map3_v1:.4f}')
print(f'   Submission saved!')

EXPERIMENT: baseline_v1
  Text strategy : v1_simple
  TF-IDF features: 5000


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/codespace/.netrc.


  [train] ACC=1.0000  F1=1.0000  MAP@3=1.0000
  [val] ACC=1.0000  F1=1.0000  MAP@3=1.0000


wandb: Currently logged in as: 23f2003236 (23f2003236-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


tfidf/features,▁
train/accuracy,▁
train/f1,▁
train/map_at_3,▁
val/accuracy,▁
val/f1,▁
val/map_at_3,▁
tfidf/features,5000
train/accuracy,1
train/f1,1
train/map_at_3,1


  WandB run: baseline_v1

Experiment 1 done! Val MAP@3 = 1.0000
   Submission saved!


## 8. Experiment 2 — v2 repeated prompt

**Hypothesis:** Repeating the question before each option gives TF-IDF more
signal about the Q-A relationship (each option now co-occurs with the question
tokens in a tighter window).

Also bumps vocab to 10K and `C` to 5.0 (less regularization).

In [20]:


config_v2 = {
    'text_strategy': 'v2_repeated',   # prompt before every option
    'max_features' : 10000,           # 2x more vocab
    'min_df'       : 3,               # include rarer words
    'max_df'       : 0.85,
    'ngram_range'  : [1, 2],
    'lr_C'         : 5.0,             # less regularization from v1
    'description'  : 'v2 - repeated prompt + larger vocab + higher C'
}

model_v2, tfidf_v2, val_map3_v2, sub_v2 = run_experiment(
    run_name = 'optimized_v2_repeated_prompt',
    config   = config_v2,
    train_df = train_df,
    val_df   = val_df,
    test_df  = test_df
)

sub_v2.to_csv(OUTPUT_DIR / 'predictions' / 'v2_submission.csv', index=False)
print(f'\n Experiment 2 done! Val MAP@3 = {val_map3_v2:.4f}')

EXPERIMENT: optimized_v2_repeated_prompt


  Text strategy : v2_repeated
  TF-IDF features: 10000
  [train] ACC=1.0000  F1=1.0000  MAP@3=1.0000
  [val] ACC=1.0000  F1=1.0000  MAP@3=1.0000


tfidf/features,▁
train/accuracy,▁
train/f1,▁
train/map_at_3,▁
val/accuracy,▁
val/f1,▁
val/map_at_3,▁
tfidf/features,10000
train/accuracy,1
train/f1,1
train/map_at_3,1


  WandB run: optimized_v2_repeated_prompt

 Experiment 2 done! Val MAP@3 = 1.0000


## 9. Experiment 3 — v3 labeled options + trigrams

**Hypothesis:** Explicitly labeling `option a:`, `option b:` etc. helps
TF-IDF learn option-specific patterns. Combined with trigrams and a 15K
vocab to capture richer local context.

In [21]:

config_v3 = {
    'text_strategy': 'v3_labeled',    # 'option a: ...' labels
    'max_features' : 15000,           # 3x vocab
    'min_df'       : 2,               # very rare words too
    'max_df'       : 0.9,
    'ngram_range'  : [1, 3],          # include trigrams too
    'lr_C'         : 3.0,
    'description'  : 'v3 - labeled options + trigrams + large vocab'
}

model_v3, tfidf_v3, val_map3_v3, sub_v3 = run_experiment(
    run_name = 'optimized_v3_labeled_trigrams',
    config   = config_v3,
    train_df = train_df,
    val_df   = val_df,
    test_df  = test_df
)

sub_v3.to_csv(OUTPUT_DIR / 'predictions' / 'v3_submission.csv', index=False)
print(f'\n Experiment 3 done! Val MAP@3 = {val_map3_v3:.4f}')

EXPERIMENT: optimized_v3_labeled_trigrams
  Text strategy : v3_labeled


  TF-IDF features: 15000
  [train] ACC=1.0000  F1=1.0000  MAP@3=1.0000
  [val] ACC=1.0000  F1=1.0000  MAP@3=1.0000


tfidf/features,▁
train/accuracy,▁
train/f1,▁
train/map_at_3,▁
val/accuracy,▁
val/f1,▁
val/map_at_3,▁
tfidf/features,15000
train/accuracy,1
train/f1,1
train/map_at_3,1


  WandB run: optimized_v3_labeled_trigrams

 Experiment 3 done! Val MAP@3 = 1.0000


## 10. Save best results

Pick the experiment with the highest validation MAP@3 and persist:
- The best submission CSV as `BEST_submission.csv`
- The best model + TF-IDF vectorizer as pickle files (for later ensembling)

In [22]:

results = {
    'v1_baseline'  : val_map3_v1,
    'v2_repeated'  : val_map3_v2,
    'v3_labeled'   : val_map3_v3,
}

In [23]:
best_name  = max(results, key=results.get)
best_score = results[best_name]

# Save best submission
best_subs = {'v1_baseline': sub_v1, 'v2_repeated': sub_v2, 'v3_labeled': sub_v3}
best_sub  = best_subs[best_name]
best_path = OUTPUT_DIR / 'predictions' / 'BEST_submission.csv'
best_sub.to_csv(best_path, index=False)
print(f'\n Best submission saved {best_path}')


 Best submission saved ../outputs/predictions/BEST_submission.csv


In [24]:
# Save the best model + its TF-IDF vectorizer for later ensembling
best_models = {
    'v1_baseline': (model_v1, tfidf_v1),
    'v2_repeated': (model_v2, tfidf_v2),
    'v3_labeled' : (model_v3, tfidf_v3)
}
best_model, best_tfidf = best_models[best_name]

model_dir = OUTPUT_DIR / 'models'
with open(model_dir / 'tfidf_best_lr_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open(model_dir / 'tfidf_best_vectorizer.pkl', 'wb') as f:
    pickle.dump(best_tfidf, f)